# DSAI 413 — A2: Chest X-Ray Intelligence System
### Kaggle Notebook — Full Pipeline

**Before running:**
1. Add the MIMIC-CXR dataset: right panel → Add data → search `mimic-cxr-dataset` by simhadrisadaram
2. Enable GPU: right panel → Session options → Accelerator → GPU P100
3. Add your secrets: right panel → Add-ons → Secrets → add `HF_TOKEN` and `GROQ_API_KEY`

**Sections:**
1. Setup
2. Data loading
3. QA dataset creation
4. Mode 1 — Report Generation
5. Build FAISS index
6. Mode 2 — QA (RAG)
7. Model Comparison
8. Launch Streamlit demo

## 1. Setup

In [ ]:
# Clone the project repo
!git clone https://github.com/nourrosama/DSAI-413-A2.git
%cd DSAI-413-A2

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q

In [ ]:
# Set API keys from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"]     = secrets.get_secret("HF_TOKEN")
os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HuggingFace login OK")

In [ ]:
# Auto-discover dataset paths from Kaggle-mounted input
import os, sys

sys.path.insert(0, '/kaggle/working/DSAI-413-A2')

# Walk /kaggle/input to find the CSV and image directory
_csv_path = None
_img_base = None

for root, dirs, files in os.walk('/kaggle/input'):
    for fname in files:
        if fname == 'mimic_cxr_aug_train.csv' and _csv_path is None:
            _csv_path = os.path.join(root, fname)
    for d in dirs:
        if d == 'official_data_iccv_final' and _img_base is None:
            _img_base = os.path.join(root, d)

if _csv_path is None:
    # Fallback: list everything so we can debug
    print("=== /kaggle/input contents ===")
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth > 2: continue
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/")
        if depth <= 1:
            for f in files[:5]:
                print(f"{indent}  {f}")
    raise FileNotFoundError("Could not find mimic_cxr_aug_train.csv in /kaggle/input — check dataset is added correctly.")

import src.config as cfg
cfg.MIMIC_CSV_PATH = _csv_path
cfg.MIMIC_IMG_BASE = _img_base or os.path.dirname(_csv_path)

print("Config updated")
print("CSV :", cfg.MIMIC_CSV_PATH)
print("Base:", cfg.MIMIC_IMG_BASE)


In [ ]:
# Verify GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM         :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Data Loading

In [ ]:
from src.preprocessing import load_mimic_subset

images, reports, img_paths = load_mimic_subset(
    cfg.MIMIC_CSV_PATH,
    img_col=cfg.MIMIC_IMG_COL,
    text_col=cfg.MIMIC_TEXT_COL,
    subset_size=cfg.DATASET_SUBSET_SIZE,
    image_base_dir=cfg.MIMIC_IMG_BASE,
)

print(f'Loaded {len(images)} image-report pairs')
print(f'Sample report: {reports[0][:100]}...')

In [ ]:
# Train / val / test split
import json, os
from sklearn.model_selection import train_test_split

indices = list(range(len(images)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

os.makedirs('data', exist_ok=True)
with open('data/split_indices.json', 'w') as f:
    json.dump({'train': train_idx, 'val': val_idx, 'test': test_idx}, f)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

In [ ]:
# Visualize sample images
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(images[i], cmap='gray')
    ax.set_title(f'Sample {i+1}', fontsize=9)
    ax.axis('off')
plt.suptitle('MIMIC-CXR Sample Images', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. QA Dataset Creation

In [ ]:
import os, json
from src.qa_dataset_creation import build_qa_dataset

os.makedirs('data/qa_dataset', exist_ok=True)

if os.path.exists(cfg.QA_PAIRS_PATH):
    with open(cfg.QA_PAIRS_PATH) as f:
        qa_pairs = json.load(f)
    print(f'Loaded existing QA dataset: {len(qa_pairs)} pairs')
else:
    print('Building QA dataset via Groq...')
    qa_pairs = build_qa_dataset(
        reports=reports,
        image_paths=img_paths,
        target_size=cfg.QA_TOTAL_TARGET,
    )

# Preview
for p in qa_pairs[:3]:
    print(f'  Q: {p["question"]}')
    print(f'  A: {p["answer"]}')
    print(f'  Category: {p["category"]}\n')

## 4. Mode 1 — Report Generation

In [ ]:
from src.mode1_report_gen import ReportGenerationPipeline, PROMPT_VARIANTS

pipeline_mode1 = ReportGenerationPipeline(use_clip=True, medgemma_load_in_4bit=True)
pipeline_mode1.load_models()
print('Mode 1 pipeline ready')

In [ ]:
# Generate report for a single test image
test_image  = images[test_idx[0]]
test_report = reports[test_idx[0]]

result = pipeline_mode1.run(image=test_image, ground_truth_report=test_report)

print('=== GENERATED REPORT ===')
print(result['report'])
print('\n=== GROUND TRUTH ===')
print(test_report)
print('\n=== CLIP ALIGNMENT SCORE ===')
print(result['clip_alignment'])
print('\n=== METRICS ===')
print(result['metrics'])

In [ ]:
# Show the X-ray alongside the report
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(test_image, cmap='gray')
axes[0].set_title('Input X-Ray', fontweight='bold')
axes[0].axis('off')

axes[1].axis('off')
axes[1].text(0.02, 0.98, result['report'], transform=axes[1].transAxes,
             fontsize=8, verticalalignment='top', wrap=True,
             bbox=dict(boxstyle='round', facecolor='#1e1e2e', alpha=0.8),
             color='white')
axes[1].set_title('Generated Report (MedGemma)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Batch evaluation on test set
from src.evaluation import compute_report_metrics

n_eval = min(20, len(test_idx))
eval_images  = [images[i]  for i in test_idx[:n_eval]]
eval_reports = [reports[i] for i in test_idx[:n_eval]]

batch_results = pipeline_mode1.run_batch(eval_images, ground_truth_reports=eval_reports)
gen_reports   = [r['report'] for r in batch_results]

medgemma_metrics = compute_report_metrics(gen_reports, eval_reports)
print('\n=== MedGemma Report Metrics ===')
for k, v in medgemma_metrics.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Prompt variant comparison
import matplotlib.pyplot as plt

variant_metrics = {}
n_compare = 5

for name, prompt in PROMPT_VARIANTS.items():
    gen = [pipeline_mode1._medgemma.generate_report(img, prompt=prompt) for img in eval_images[:n_compare]]
    m = compute_report_metrics(gen, eval_reports[:n_compare])
    variant_metrics[name] = m
    print(f'{name:12s}: ROUGE-L={m["rouge_l"]}, BLEU-1={m["bleu1"]}')

variants = list(variant_metrics.keys())
rouge_l  = [variant_metrics[v]['rouge_l'] for v in variants]
bleu1    = [variant_metrics[v]['bleu1']   for v in variants]
x = range(len(variants))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([xi - 0.2 for xi in x], rouge_l, 0.35, label='ROUGE-L', color='#0d6efd')
ax.bar([xi + 0.2 for xi in x], bleu1,   0.35, label='BLEU-1',  color='#28a745')
ax.set_xticks(x)
ax.set_xticklabels(variants, rotation=20)
ax.set_title('Prompt Variant Comparison (MedGemma)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Build FAISS Index (ColPali)

In [ ]:
import os
from src.retrieval import build_index_from_images, RetrievalIndex

train_images  = [images[i]    for i in train_idx]
train_reports = [reports[i]   for i in train_idx]
train_paths   = [img_paths[i] for i in train_idx]

if os.path.exists(cfg.FAISS_INDEX_PATH):
    colpali_index = RetrievalIndex.load()
    print(f'Loaded existing index: {colpali_index.index.ntotal} vectors')
else:
    print('Building ColPali FAISS index...')
    colpali_index = build_index_from_images(
        images=train_images,
        reports=train_reports,
        image_paths=train_paths,
        backend='colpali',
        save=True,
    )
    print(f'Index built: {colpali_index.index.ntotal} vectors')

In [ ]:
# Build CLIP index for comparison
from src.retrieval import build_index_from_images

clip_index_path = cfg.FAISS_INDEX_PATH.replace('.bin', '_clip.bin')
clip_meta_path  = cfg.FAISS_META_PATH.replace('.json', '_clip.json')

if os.path.exists(clip_index_path):
    clip_index = RetrievalIndex.load(clip_index_path, clip_meta_path)
    print(f'Loaded CLIP index: {clip_index.index.ntotal} vectors')
else:
    print('Building CLIP FAISS index...')
    clip_index = build_index_from_images(
        images=train_images,
        reports=train_reports,
        image_paths=train_paths,
        backend='clip',
        save=False,
    )
    clip_index.save(clip_index_path, clip_meta_path)
    print(f'CLIP index built: {clip_index.index.ntotal} vectors')

## 6. Mode 2 — QA (RAG)

In [ ]:
from src.mode2_qa import QAPipeline

qa_pipeline = QAPipeline(
    index=colpali_index,
    retrieval_backend='colpali',
    top_k=cfg.FAISS_TOP_K,
    medgemma_load_in_4bit=True,
)
qa_pipeline.load_models()
print('Mode 2 pipeline ready')

In [ ]:
# Run QA on a sample image
sample_image    = images[test_idx[0]]
sample_question = 'Is there any evidence of pleural effusion?'

result_qa = qa_pipeline.run(image=sample_image, question=sample_question)

print(f'Question : {result_qa["question"]}')
print(f'Answer   : {result_qa["answer"]}')
print(f'\nRetrieved context (top result):')
print(result_qa['retrieved_results'][0]['report'][:300])

In [ ]:
# Visualize retrieval scores
retrieved = result_qa['retrieved_results']
ranks  = [f'Rank {i+1}' for i in range(len(retrieved))]
scores = [r['score'] for r in retrieved]

plt.figure(figsize=(8, 4))
bars = plt.bar(ranks, scores, color='teal', edgecolor='white')
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{score:.3f}', ha='center', fontsize=9)
plt.title(f'ColPali Retrieval Scores\nQ: "{sample_question}"')
plt.ylabel('Cosine Similarity')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Batch QA evaluation
from src.preprocessing import load_image
from src.evaluation import compute_qa_metrics

eval_qa = qa_pairs[:30]
preds_rag, preds_no_rag, gts = [], [], []

for pair in eval_qa:
    try:
        img = load_image(pair['image_path'])
        # With RAG
        r = qa_pipeline.run(image=img, question=pair['question'])
        preds_rag.append(r['answer'])
        # Without RAG
        ans = qa_pipeline._medgemma.answer_question(image=img, question=pair['question'], context='')
        preds_no_rag.append(ans)
        gts.append(pair['answer'])
    except Exception as e:
        print(f'  Error: {e}')

metrics_rag    = compute_qa_metrics(preds_rag,    gts)
metrics_no_rag = compute_qa_metrics(preds_no_rag, gts)

print('\n=== QA Evaluation ===')
print(f'With RAG (ColPali):  EM={metrics_rag["exact_match"]:.4f}, F1={metrics_rag["token_f1"]:.4f}')
print(f'Without RAG:         EM={metrics_no_rag["exact_match"]:.4f}, F1={metrics_no_rag["token_f1"]:.4f}')

## 7. Model Comparison

In [ ]:
# Retrieval comparison: ColPali vs CLIP Precision@K
from src.models.colpali import ColPaliModel
from src.models.clip_model import CLIPEncoder
from src.evaluation import compute_precision_at_k

colpali_model = ColPaliModel()
colpali_model.load()

clip_enc = pipeline_mode1._clip

n_eval_ret = 20
colpali_retrieved_list, clip_retrieved_list, relevant_list = [], [], []

for i in test_idx[:n_eval_ret]:
    img     = images[i]
    gt_path = img_paths[i]

    q_cp = colpali_model.embed_query_image(img)
    colpali_retrieved_list.append(colpali_index.query(q_cp, top_k=5))

    q_cl = clip_enc.embed_single_image(img)
    clip_retrieved_list.append(clip_index.query(q_cl, top_k=5))

    relevant_list.append([gt_path])

colpali_pak = compute_precision_at_k(colpali_retrieved_list, relevant_list, k=5)
clip_pak    = compute_precision_at_k(clip_retrieved_list,    relevant_list, k=5)

print(f'ColPali Precision@5 : {colpali_pak:.4f}')
print(f'CLIP    Precision@5 : {clip_pak:.4f}')

In [ ]:
# Final comparison table
from src.evaluation import generate_comparison_table
from IPython.display import display, Markdown

all_model_results = {
    'MedGemma (Report Gen)': {
        'task': 'Report Generation',
        **medgemma_metrics,
    },
    'ColPali (Retrieval)': {
        'task': 'Retrieval',
        'precision_at_k': colpali_pak,
    },
    'CLIP (Retrieval)': {
        'task': 'Retrieval',
        'precision_at_k': clip_pak,
    },
    'MedGemma + ColPali RAG': {
        'task': 'QA with RAG',
        'exact_match': metrics_rag['exact_match'],
        'token_f1':    metrics_rag['token_f1'],
    },
    'MedGemma (no RAG)': {
        'task': 'QA without RAG',
        'exact_match': metrics_no_rag['exact_match'],
        'token_f1':    metrics_no_rag['token_f1'],
    },
}

table_md = generate_comparison_table(all_model_results)
display(Markdown(table_md))

In [ ]:
# Visualize comparison
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Report metrics
metrics_names = ['BLEU-1', 'BLEU-4', 'ROUGE-L']
metrics_vals  = [medgemma_metrics.get('bleu1', 0),
                 medgemma_metrics.get('bleu4', 0),
                 medgemma_metrics.get('rouge_l', 0)]
axes[0].bar(metrics_names, metrics_vals, color='#0d6efd')
axes[0].set_title('MedGemma — Report Generation')
axes[0].set_ylim(0, 1)

# Retrieval P@K
axes[1].bar(['ColPali', 'CLIP'], [colpali_pak, clip_pak], color=['teal', 'coral'])
axes[1].set_title('Retrieval — Precision@5')
axes[1].set_ylim(0, 1)

# QA F1
axes[2].bar(['With RAG', 'No RAG'],
            [metrics_rag['token_f1'], metrics_no_rag['token_f1']],
            color=['#28a745', '#dc3545'])
axes[2].set_title('QA — Token F1')
axes[2].set_ylim(0, 1)

plt.suptitle('Model Comparison Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Launch Streamlit Demo

In [ ]:
# Install tunnel
!pip install pyngrok -q
from pyngrok import conf, ngrok

# If you have an ngrok token add it here, otherwise use localtunnel below
# ngrok.set_auth_token('your_ngrok_token')

In [ ]:
import subprocess, threading, time

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'app/app.py',
        '--server.port=8501',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false'
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()
time.sleep(15)

# Option A — localtunnel (no account needed)
!npx localtunnel --port 8501

# Option B — if you have ngrok token, comment out Option A and use:
# public_url = ngrok.connect(8501)
# print('URL:', public_url)